# Simple RAG Pipeline

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
import chromadb

# Create persistent client (saves to disk)
client = chromadb.Client()

collection = client.create_collection(name="rag_collection")

In [ ]:
documents = [
    "Machine learning is a field of artificial intelligence.",
    "Neural networks are used in deep learning.",
    "The capital of India is New Delhi.",
    "Cricket is a popular sport in India."
]

# Generate embeddings
embeddings = model.encode(documents).tolist()

# Add to Chroma
collection.add(
    documents=documents,
    embeddings=embeddings,
    ids=[str(i) for i in range(len(documents))]
)

In [ ]:
query = "What is deep learning?"

query_embedding = model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=2
)

print(results["documents"])
print(results["distances"])

In [ ]:
def retrieve(query, top_k=1):
    query_embedding = model.encode(query).tolist()
    
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )
    
    return results["documents"][0]


In [ ]:
query = "What is deep learning?"
retrieved_docs = retrieve(query)
print(retrieved_docs)

In [ ]:
query = "What is the capital of India?"
retrieved_docs = retrieve(query)    
print(retrieved_docs)

# Hugging Face - Langchain

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
from langchain_community.vectorstores import Chroma

persist_directory = "./chroma_db"

vectorstore = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding_model
)

In [ ]:
from langchain_core.documents import Document

documents = [
    Document(page_content="Machine learning is a field of artificial intelligence."),
    Document(page_content="Neural networks are used in deep learning."),
    Document(page_content="The capital of India is New Delhi."),
    Document(page_content="Cricket is a popular sport in India.")
]

vectorstore.add_documents(documents)

vectorstore.persist()

In [ ]:
query = "What is deep learning?"

results = vectorstore.similarity_search(query, k=2)

for doc in results:
    print(doc.page_content)

In [ ]:
docs_with_score = vectorstore.similarity_search_with_score(
    "Explain neural networks",
    k=2
)

for doc, score in docs_with_score:
    print(score, doc.page_content)

In [ ]:
query_embedding = embedding_model.embed_query("Transformers in NLP")

docs = vectorstore.similarity_search_by_vector(
    query_embedding,
    k=2
)
for doc in docs:
    print(doc.page_content)

In [ ]:
docs = vectorstore.max_marginal_relevance_search(
    "Explain attention mechanism",
    k=2,
    fetch_k=10
)
for doc in docs:
    print(doc.page_content)

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

docs = retriever.invoke("Explain transformers")
for doc in docs:
    print(doc.page_content)

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "score_threshold": -0.9,
        "k": 2
    }
)
docs = retriever.invoke("Explain transformers")
for doc in docs:
    print(doc.page_content)